# Explore OpenAlex derived tables

1. `openalex-load`
2. `uv run python scripts/fetch.py`
3. `uv run python scripts/compile.py --duckdb`
4. Point this notebook at your project's kernel


In [ ]:
import sys
from pathlib import Path

# Kernel cwd is often notebooks/ — walk up to the project root.
here = Path.cwd().resolve()
root = next(
    (
        p
        for p in [here, *here.parents]
        if (p / "config.toml").is_file() and (p / "src").is_dir()
    ),
    None,
)
if root is None:
    raise FileNotFoundError("config.toml not found; open this repo as the VS Code folder")
sys.path.insert(0, str(root / "src"))

from paths import load_paths

import polars as pl

p = load_paths()
works = pl.read_parquet(p.derived / "works.parquet")
works.head()


In [ ]:
# Optional SQL over Parquet via DuckDB
import duckdb

db = p.derived / "catalog.duckdb"
if db.is_file():
    con = duckdb.connect(str(db), read_only=True)
    print(con.execute("SELECT publication_year, COUNT(*) AS n FROM works GROUP BY 1 ORDER BY 1").fetchdf())
else:
    print("No catalog.duckdb — re-run: uv run python scripts/compile.py --duckdb")
